# Import BSF cell ranger output and save as Seurat object

In [ ]:
library_load <- suppressMessages(
    
    list(
        
        # Seurat 
        library(Seurat),

        # Module score 
        library(UCell), 
        
        # Data 
        library(dplyr), 
        
        # Pyhton compatibility
        library(reticulate)
        
    )
)

In [ ]:
# Configure reticulate 
use_condaenv(condaenv='p.3.10.16-FD20250213HSCT', conda="/nobackup/peer/fdeckert/miniconda3/bin/conda", required=NULL)
py_config()

In [ ]:
# Set working directory to project root
setwd("/research/peer/fdeckert/FD20250213HSCT")

# Parameter settings

In [ ]:
# Seurat parameters 
min_cells    <- 0 # Filter out features which are only present in less than min_cells cells - default 3
min_features <- 0 # Filter out cells which have fewer than min_features features - default 200

# Files
h5ad_file <- "data/object/raw.h5ad"

# Project folder

## HSCT samples

In [ ]:
sample_path <- "/research/lab_stary/sequencing_projects/hsct/COUNT/"
sample_dir <- list.dirs(path=sample_path, full.names=FALSE, recursive=FALSE) 

In [ ]:
sample_dir_1 <- sample_dir %>% 
    subset(., grepl("^S", .)) %>% 
    subset(., grepl("*transcriptome", .))
sample_path_1 <- paste0(sample_path, sample_dir_1)
patient_name_1 <- sapply(strsplit(sample_dir_1,"_"), `[`, 2) %>% substr(start=1 , stop=3)
time_point_1 <- sapply(strsplit(sample_dir_1,"_"), `[`, 3) %>% 
    gsub("baseline", "Baseline", .) %>% 
    gsub("day0", "Tx", .) %>% 
    gsub("day14", "D14", .) %>% 
    gsub("day100", "D100", .)
gvhd_status_1 <- rep("HSCT", length(sample_dir_1))

In [ ]:
sample_dir_2 <- sample_dir %>% 
    subset(., grepl("^HSCT", .)) %>% 
    subset(., grepl("*transcriptome", .))
sample_path_2 <- paste0(sample_path, sample_dir_2)
patient_name_2 <- sapply(strsplit(sample_dir_2,"_"), `[`, 2) %>% substr(start=1 , stop=3)
time_point_2 <- sapply(strsplit(sample_dir_2,"_"), `[`, 2) %>% substr(start=4 , stop=4) %>% 
    gsub("\\b1\\b", "Baseline", .) %>% 
    gsub("\\b2\\b", "Tx", .) %>% 
    gsub("\\b3\\b", "D14", .) %>% 
    gsub("\\b4\\b", "D100", .)
gvhd_status_2 <- rep("HSCT", length(sample_dir_2))

## aGVHD samples

In [ ]:
sample_path <- "/research/lab_stary/sequencing_projects/acute_gvhd/COUNT/"
sample_dir <- list.dirs(path=sample_path, full.names=FALSE, recursive=FALSE) 

In [ ]:
sample_dir_3 <- sample_dir %>% 
    subset(., grepl("^CoR", .)) %>% 
    subset(., grepl("*transcriptome", .))
sample_path_3 <- paste0(sample_path, sample_dir_3)
patient_name_3 <- sapply(strsplit(sample_dir_3,"_"), `[`, 3) %>% substr(start=1 , stop=3)
time_point_3 <- rep("GVHD", length(sample_dir_3))
gvhd_status_3 <- rep("aGVHD", length(sample_dir_3))

In [ ]:
sample_dir_4 <- sample_dir %>% 
    subset(., grepl("^GvD", .)) %>% 
    subset(., grepl("*transcriptome", .))
sample_path_4 <- paste0(sample_path, sample_dir_4)
patient_name_4 <- sapply(strsplit(sample_dir_4,"_"), `[`, 2) %>% substr(start=1 , stop=3)
time_point_4 <- rep("GVHD", length(sample_dir_4))
gvhd_status_4 <- rep("aGVHD", length(sample_dir_4))

## Combine meta data

In [ ]:
meta <- data.frame(
  
    sample_path=c(sample_path_1, sample_path_2, sample_path_3, sample_path_4), 
    sample_dir=c(sample_dir_1, sample_dir_2, sample_dir_3, sample_dir_4), 
    patient_name=c(patient_name_1, patient_name_2, patient_name_3, patient_name_4), 
    time_point=c(time_point_1, time_point_2, time_point_3, time_point_4), 
    gvhd_status=c(gvhd_status_1, gvhd_status_2, gvhd_status_3, gvhd_status_4)
  
) 

## Remove samples 

In [ ]:
meta <- meta[meta$sample_dir != "CoR_GvD_05_transcriptome", ]

## Add meta data

In [ ]:
# digestion_protocol
meta$digestion_protocol <- ifelse(meta$sample_dir %in% sample_dir_1, "old", "new")

In [ ]:
# Rename patient 13
meta$patient_name <- ifelse(meta$patient_name=="13", "117", meta$patient_name)

In [ ]:
# patient_id

# Set patient dummy patient idx to match patient names
meta$patient_tmp <- meta$patient_name 
meta$patient_tmp <- ifelse(meta$patient_tmp=="116", "784", meta$patient_tmp)

meta <- meta %>%  dplyr::mutate(patient_id=paste0("Patient_", match(patient_tmp, unique(patient_tmp))))

meta$patient_tmp <- NULL

In [ ]:
# sample_id 
meta$sample_name <- paste0(meta$patient_id, "_", meta$time_point)

## Write meta csv

In [ ]:
write.csv(meta, "meta.csv", row.names=FALSE)

# Import filtered count matrix as Seurat object

In [ ]:
bsf_import <- function(meta, type) {
    
    # Import function
    import_cnt <- function(sample_meta, type){

        message(paste("import:", paste0(sample_meta$sample_path, "/", type)))

        # Read 10x data in as sparse matrix (sm)
        import_10x <- Read10X(data.dir=paste0(sample_meta$sample_path, "/", type))

        # Get count matrix and ignore HTO counts 
        cnt <- import_10x[[1]]

        # Meta data 
        meta <- sapply(sample_meta, rep.int, times=ncol(cnt)) %>% data.frame()
        rownames(meta) <- colnames(cnt)

        # Transform sparse matrix to seurat object (so)
        so <- CreateSeuratObject(counts=cnt, meta.data=meta, min.cells=min_cells, min.features=min_features)

        # HTO demux information
        hto_demux_class <- read.csv(paste0(sample_meta$sample_path, "/", "HTO_demux.csv"), row.names=1)
        rownames(hto_demux_class) <- paste0(rownames(hto_demux_class), "-1")
        colnames(hto_demux_class) <- "hto_demux_class"
        so <- AddMetaData(so, hto_demux_class)

        # Set ident
        so$orig.ident <- NULL

        return(so)

    }

    # Import count data with HTO demux class
    so <- lapply(split(meta, f=meta$sample_path), import_cnt, list(type=type))

    # Merge Seurat objects 
    so <- merge(x=so[[1]], y=so[2:length(so)])

    # Merge layers 
    so <- JoinLayers(object=so, layers="counts")

    # Set cell_id 
    so$cell_id <- colnames(so)

    # Order meta data and drop categories
    so@meta.data <- so@meta.data[, c("sample_path", "cell_id", "sample_dir", "patient_name", "patient_id", "time_point", "gvhd_status", "sample_name", "digestion_protocol", "hto_demux_class")]

    # Complete HTO demux class 
    so$hto_demux_class <- ifelse(so$hto_demux_class %in% c("HTO-blood", "HTO-Blood", "HTO-blood_1"), "Blood", so$hto_demux_class)
    so$hto_demux_class <- ifelse(so$hto_demux_class %in% c("HTO-skin", "HTO-Skin", "HTO-skin_1"), "Skin", so$hto_demux_class)

    # Exclude samples from patient name 13 which are HTO-blood_2 und HTO-skin_2
    so <- subset(so, subset=hto_demux_class %in% c("Blood", "Skin", "Doublet", "Negative"))

    return(so)

}

# Save filtered h5ad

In [ ]:
so <- bsf_import(meta, "filtered_feature_bc_matrix")

# CellCycle score 

In [ ]:
# Seurat cell cycle genes 
cc_genes_s <- cc.genes.updated.2019$s.genes
cc_genes_g2m <- cc.genes.updated.2019$g2m.genes

In [ ]:
so <- UCell::AddModuleScore_UCell(so, features=list(S_ucell=cc_genes_s, G2M_ucell=cc_genes_g2m), assay="RNA", slot="counts", name="")

In [ ]:
# Store data as h5ad 
ad <- import("anndata")
pd <- import("pandas")
np <- import("numpy")
sp <- import("scipy")
    
# Transform dgCMatrix to sparse sc_sparse matrix
X <- GetAssayData(so, assay="RNA", layer="counts") %>% t()

# Combine 
adata <- ad$AnnData(X=sp$sparse$csr_matrix(X), obs=so@meta.data)
adata$var_names <- rownames(so)

# Save
adata$write_h5ad(h5ad_file)

# Session info 

In [ ]:
sessionInfo()